In [3]:
!pip install unsloth transformers datasets accelerate peft trl bitsandbytes
!pip install pdfplumber pymupdf
!pip install scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.3/65.3 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 119.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.2/421.2 kB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 123.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 130.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 117.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

In [2]:
import torch
print(torch.cuda.is_available())

True


In [3]:
from unsloth import FastLanguageModel

model,tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/llama-3-8b-bnb-4bit",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.6: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3-8b-bnb-4bit as a legacy tokenizer.


In [4]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,  # rank
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
)

Not an error, but Unsloth cannot patch MLP layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Unsloth 2026.4.6 patched 32 layers with 32 QKV layers, 32 O layers and 0 MLP layers.


In [3]:
!rm -rf ~/.cache/huggingface/datasets/theatticusproject___cuad

In [18]:
from google.colab import files
uploaded = files.upload()

Saving CUAD_v1.json to CUAD_v1.json


In [6]:
import json

with open("CUAD_v1.json", "r") as f:
    data = json.load(f)

In [7]:
examples = []

for contract in data["data"]:
    for para in contract["paragraphs"]:
        context = para["context"]

        for qa in para["qas"]:
            question = qa["question"]

            if qa["is_impossible"] or len(qa["answers"]) == 0:
                answer = "No clause found"
            else:
                answer = qa["answers"][0]["text"]

            examples.append({
                "context": context,
                "question": question,
                "answer": answer
            })

In [6]:
print(examples[:5])

[{'context': 'EXHIBIT 10.6\n\n                              DISTRIBUTOR AGREEMENT\n\n         THIS  DISTRIBUTOR  AGREEMENT (the  "Agreement")  is made by and between Electric City Corp.,  a Delaware  corporation  ("Company")  and Electric City of Illinois LLC ("Distributor") this 7th day of September, 1999.\n\n                                    RECITALS\n\n         A. The  Company\'s  Business.  The Company is  presently  engaged in the business  of selling an energy  efficiency  device,  which is  referred to as an "Energy  Saver"  which may be improved  or  otherwise  changed  from its present composition (the "Products").  The Company may engage in the business of selling other  products  or  other  devices  other  than  the  Products,  which  will be considered  Products if Distributor  exercises its options pursuant to Section 7 hereof.\n\n         B. Representations.  As an inducement to the Company to enter into this Agreement,  the  Distributor  has  represented  that  it has 

In [8]:
def format_example(e):
    return {
        "text": f"""### Instruction:
{e['question']}

### Input:
{e['context']}

### Response:
{e['answer']}"""
    }

formatted_data = [format_example(e) for e in examples]

In [9]:
from datasets import Dataset

dataset = Dataset.from_list(formatted_data)

In [11]:
dataset = dataset.shuffle(seed=42).select(range(2000))

In [12]:
import pandas as pd

df = pd.DataFrame(dataset[:5])
df

,text
0,### Instruction:\nHighlight the parts (if any)...
1,### Instruction:\nHighlight the parts (if any)...
2,### Instruction:\nHighlight the parts (if any)...
3,### Instruction:\nHighlight the parts (if any)...
4,### Instruction:\nHighlight the parts (if any)...


In [13]:
print(len(dataset))

2000


In [14]:
print(dataset.column_names)

['text']


In [19]:
def is_valid(example):
    text = example["text"]

    # must contain proper structure
    return (
        "### Instruction:" in text and
        len(text) > 50 and
        "### Input" in text   # ensure not cut
    )

dataset = dataset.filter(is_valid)

Filter:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [20]:
dataset = dataset.filter(lambda x: not x["text"].endswith("..."))

Filter:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [21]:
dataset = dataset.select(range(1000))

In [2]:
for i in range(10):
    print(dataset[i]["text"][:500])

NameError: name 'dataset' is not defined

In [1]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        max_steps=100,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=10,
        output_dir="outputs",
        dataloader_num_workers=0,   # 🔥 IMPORTANT
    ),
)

trainer.train()

NameError: name 'model' is not defined

In [ ]:
model.save_pretrained("lora-contract-model")
tokenizer.save_pretrained("lora-contract-model")

In [ ]:
FastLanguageModel.for_inference(model)

prompt = """
### Instruction:
Extract governing law clause

### Input:
This agreement shall be governed by the laws of California.

### Response:
"""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens=100)
print(tokenizer.decode(outputs[0]))